# Station census notebook
Given a rectangular bounding box and an analysis period, the notebook queries the OpenAQ database for stations in
that geographical region which measure PM2.5 and whose active measurement range is larger than or equal to the analysis period

In [15]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
# External libraries imports

import os, dotenv
from pathlib import Path
from datetime import datetime

from openaq import OpenAQ
import pandas as pd
import geopandas as gpd
from numpy import argmax

In [17]:
# Custom libraries imports

import monitoring_deserts.data.fetch_openaq as fetch_openaq


In [18]:
## Notebook level constants - later change to read from config file

# Coordinates for South Asia - India, Pakistan, Nepal, Bangladesh, Sri-Lanka + Afghanistan 
BOUNDING_BOX = BB_Xm, BB_Ym, BB_XM, BB_Ym = (60.424805,5.353521,93.383789,38.272689)
PERIOD = DAY_START, DAY_END = ["2024-01-01", "2026-01-01"] # Half-open interval

In [37]:
# Notebook starts here

dotenv.load_dotenv()
with OpenAQ() as client:
    #print("Fetching pollutant ID...")
    #pm25_id = fetch_openaq.get_pollutant_id(client, poll_name="PM25")
    #print("Fetched pollutant ID")
    #print("Fetching stations list...")
    #stations_df, stations_list = fetch_openaq.pull_stations(client, BOUNDING_BOX, pollutant_id=pm25_id)
    #print("Fetched stations list")
    #print("Built stations DataFrame...")
    print("Fetching active sensors list...")
    sensors_list = fetch_openaq.get_active_sensors(client, stations_list, pm25_id, start=DAY_START, end=DAY_END)
    print("Fetched active sensors list")
    
print(f"\nThe parameter ID for PM2.5 is {pm25_id}")
print(f"Total number of stations measuring PM2.5 in the region: {len(stations_list)}")
print(f"Number of active PM2.5 sensors in the region over the period from {DAY_START}  to {DAY_END}: {len(sensors_list)}")

Fetching active sensors list...
Checking sensor number 0
Checking sensor number 50
Checking sensor number 100
Checking sensor number 150
Checking sensor number 200
Checking sensor number 250
Checking sensor number 300
Checking sensor number 350
Checking sensor number 400


ServerError: Internal Server Error

,id,name,locality,timezone,is_mobile,is_monitor,instruments,sensors,bounds,distance,datetime_first,datetime_last,country.id,country.code,country.name,owner.id,owner.name,provider.id,provider.name,coordinates.latitude,coordinates.longitude,datetime_first.utc,datetime_first.local,datetime_last.utc,datetime_last.local
0,12,SPARTAN - IIT Kanpur,NaN,Asia/Kolkata,False,True,"[{'id': 2, 'name': 'Government Monitor'}]","[{'id': 23, 'name': 'pm25 µg/m³', 'parameter':...","(80.233, 26.519, 80.233, 26.519)",None,NaN,NaN,9,IN,India,4,Unknown Governmental Organization,226,Spartan,26.519000,80.233000,NaN,NaN,NaN,NaN
1,13,"Delhi Technological University, Delhi - CPCB",NaN,Asia/Kolkata,False,True,"[{'id': 2, 'name': 'Government Monitor'}]","[{'id': 13866, 'name': 'no2 µg/m³', 'parameter...","(77.12, 28.744, 77.12, 28.744)",None,NaN,NaN,9,IN,India,4,Unknown Governmental Organization,168,CPCB,28.744000,77.120000,2016-11-02T19:00:00Z,2016-11-03T00:30:00+05:30,2018-02-22T04:00:00Z,2018-02-22T09:30:00+05:30
2,15,IGI Airport,NaN,Asia/Kolkata,False,True,"[{'id': 2, 'name': 'Government Monitor'}]","[{'id': 27, 'name': 'co µg/m³', 'parameter': {...","(77.094, 28.56, 77.094, 28.56)",None,NaN,NaN,9,IN,India,4,Unknown Governmental Organization,168,CPCB,28.560000,77.094000,NaN,NaN,NaN,NaN
3,16,Civil Lines,NaN,Asia/Kolkata,False,True,"[{'id': 2, 'name': 'Government Monitor'}]","[{'id': 33, 'name': 'co µg/m³', 'parameter': {...","(77.2262, 28.6787, 77.2262, 28.6787)",None,NaN,NaN,9,IN,India,4,Unknown Governmental Organization,168,CPCB,28.678700,77.226200,NaN,NaN,NaN,NaN
4,17,"R K Puram, Delhi - DPCC",NaN,Asia/Kolkata,False,True,"[{'id': 2, 'name': 'Government Monitor'}, {'id...","[{'id': 12234782, 'name': 'co ppb', 'parameter...","(77.186937, 28.563262, 77.186937, 28.563262)",None,NaN,NaN,9,IN,India,8524,Delhi Pollution Control Committee,168,CPCB,28.563262,77.186937,2016-02-05T14:55:00Z,2016-02-05T20:25:00+05:30,2026-09-02T18:15:00Z,2026-09-02T23:45:00+05:30


In [ ]:
# Dump old stuff
dotenv.load_dotenv()




with OpenAQ() as oaq_client:
    pollutant_list = oaq_client.parameters.list(parameter_type="pollutant").dict()["results"]
    pm25_oaq_id = next(x["id"] for x in pollutant_list if x["name"].upper() == "PM25")
    stations_list = oaq_client.locations.list(parameters_id=pm25_oaq_id,
                                              bbox=delhi_bbox,
                                              limit=100)
    # Select only the sensors which measure PM2.5 
    pm25_sensors_id_list = [next(sensor.id for sensor in station.sensors if sensor.parameter.id == pm25_oaq_id)  for station in stations_list.results]
    # Now pull each sensor data to check when the last measurement was
    pm25_active_sensors = []
    for i, s_id in enumerate(pm25_sensors_id_list):
        sens_result = oaq_client.sensors.get(sensors_id=s_id)
        if check_last_measurement(sens_result.results[0]):
            pm25_active_sensors.append(s_id)
    # Loop to request sensor measurements. Make a first temporary DataFrame with a station-month MultiIndex 
    df_list = []
    for s_id in pm25_active_sensors:
        meas_list = oaq_client.measurements.list(sensors_id=s_id,
                                                data="days",
                                                rollup="monthly",
                                                date_from="2024-01-01",
                                                date_to="2025-12-31",
                                                limit=30)
        df_list.append(make_df(meas_list, idx=s_id))
